In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, matthews_corrcoef
from IPython.display import display
folder = Path('FINDER/Official')
scores = pd.read_csv(folder / 'daily_scores_all_markets.csv')
metrics = pd.read_csv(folder / 'market_agent_metrics.csv')
thresholds = pd.read_csv(folder / 'daily_thresholds.csv')
rows = []
for (market, agent), frame in thresholds.groupby(['market', 'agent']):
    source = scores[scores.market.eq(market)].set_index('target_date').loc[frame.target_date]
    actual = source.actual_label.to_numpy()
    probability = source[agent].to_numpy()
    predicted = (probability >= frame.threshold.to_numpy()).astype(int)
    assert np.array_equal(predicted, frame.prediction)
    row = {'market': market, 'agent': agent, 'roc_auc': roc_auc_score(actual, probability), 'ap': average_precision_score(actual, probability), 'precision': precision_score(actual, predicted, zero_division=0), 'recall': recall_score(actual, predicted, zero_division=0), 'f1': f1_score(actual, predicted, zero_division=0), 'mcc': matthews_corrcoef(actual, predicted)}
    expected = metrics[metrics.market.eq(market) & metrics.agent.eq(agent)].iloc[0]
    for field in ['roc_auc', 'ap', 'precision', 'recall', 'f1', 'mcc']:
        assert np.isclose(row[field], expected[field])
    rows.append(row)
calculated = pd.DataFrame(rows)
display(calculated)


In [ ]:
evaluate_new_run = False
run_id = 'concept'
ticker = '^GSPC'

if evaluate_new_run:
    from finder_core.settings import load_settings
    from finder_core.public_outputs import evaluate
    settings = load_settings(root, ticker=ticker)
    scores, metrics = evaluate(root / 'Runs' / ticker / run_id, root / settings['paths']['labels'])
    display(metrics)
